# Paso 3 — ML Model
Feature matrix → XGBoost + spatial block CV → flood probability raster → aggregate to ARBA parcels.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.config import load_config
from src.features import build_pixel_features, aggregate_to_parcels
from src.model import train_model, predict_raster
from src.viz import plot_raster
from pathlib import Path

cfg = load_config()
proc = Path('../data/processed')
outputs = Path('../outputs')
outputs.mkdir(exist_ok=True)

## 3.1 Build feature matrix

In [ ]:
feature_rasters = {
    'twi':           proc / 'twi.tif',
    'flow_acc':      proc / 'flow_acc.tif',
    'water_rel_elev': proc / 'water_rel_elev.tif',
    'flood_freq':    Path('../data/raw/flood_frequency.tif'),
    # Add soil type, NDVI, CHIRPS accumulations when available
}

df = build_pixel_features(
    raster_paths=feature_rasters,
    flood_target_path=Path('../data/raw/flood_frequency.tif'),
)
print(df.shape)
df.head()

## 3.2 Train with spatial block CV

In [ ]:
feature_cols = [c for c in df.columns if c != 'flooded']
model, cv_results = train_model(
    df=df,
    feature_cols=feature_cols,
    target_col='flooded',
    n_blocks=cfg['model']['n_spatial_blocks'],
    xgb_params=cfg['model']['xgboost_params'],
    output_dir=str(outputs),
)
cv_results

## 3.3 Probability raster

In [ ]:
predict_raster(
    model=model,
    feature_raster_paths=feature_rasters,
    output_path=outputs / 'flood_probability.tif',
    feature_cols=feature_cols,
    ref_raster_path=proc / 'twi.tif',
)
plot_raster(outputs / 'flood_probability.tif',
            title='Flood Probability', cmap='RdYlGn_r',
            output_path=str(outputs / 'flood_prob_map.png'))

## 3.4 Aggregate to ARBA parcels
Adjust path once catastro ARBA shapefile is downloaded.

In [ ]:
arba_path = Path('../data/raw/arba_parcelas.shp')  # download from ARBA
if arba_path.exists():
    parcel_features = aggregate_to_parcels(
        feature_raster_paths={
            'flood_prob': outputs / 'flood_probability.tif',
            'twi': proc / 'twi.tif',
        },
        parcel_shapefile=str(arba_path),
        output_path=str(outputs / 'parcel_flood_risk.gpkg'),
    )
else:
    print('ARBA shapefile not found. Download from https://www.arba.gov.ar (catastro).')